# Deformable 3D Gaussians (ingra14m/Deformable-3D-Gaussians): D-NeRF monocular benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardolanza05/dynamic-gaussian-splatting-benchmark/blob/main/notebooks/01_deformable_3dgs_dnerf.ipynb)

Part of **dynamic-gaussian-splatting-benchmark**, a controlled comparison of three dynamic Gaussian Splatting methods on the eight scenes of the monocular D-NeRF synthetic dataset, run on a free-tier Google Colab Tesla T4. The method is trained with its official training code; a benchmark monitor attached to its training loop records PSNR, SSIM, LPIPS, evaluation L1, training time, iteration count, number of Gaussians, peak VRAM and model storage into one JSON file per run.

**AI disclosure.** This notebook, the rest of the repository and part of the code used to extract the metrics (the benchmark monitor, the per-scene preparation and the analysis scripts) were generated with AI assistance (Anthropic's Claude), starting from the code of the official repository of each method. The training code of the method itself is the official one, cloned at run time; the only changes are the monitor hook appended to `train.py`, configuration overrides and environment fixes, all made by the cells below.

**Method.** Deformable 3D Gaussians for High-Fidelity Monocular Dynamic Scene Reconstruction, Yang et al., CVPR 2024 ([paper](https://arxiv.org/abs/2309.13101), [code](https://github.com/ingra14m/Deformable-3D-Gaussians)). A set of 3D Gaussians in a canonical space, deformed over time by an MLP queried per Gaussian.

Two protocols are supported:

* **Protocol A, equal iterations**: every method trains for 30 000 optimisation steps, metrics sampled every 1 000 steps.
* **Protocol B, equal quality**: training stops when the evaluation L1 reaches a per-scene target shared by the three notebooks, metrics sampled every 30 seconds.

Results, methodology and calibration are documented in the repository `README.md` and in `docs/`.

**Notebook structure**

*Part 1: configuration, setup and training*

0. User configuration and path resolution
1. Environment and repository setup
2. CUDA module fixes
3. Dataset download and per-scene preparation
4. Training with incremental benchmarking (single scene or loop, Protocol B calibration, summary)

*Part 2: evaluation and visualisation*

5. Rendering and final metrics
6. Real-time streaming viewer
7. MP4 video export

---

# Part 1: configuration, setup and training

This part is safe to run unattended. It configures the run, prepares the environment and the dataset, installs the benchmark instrumentation and trains one scene or a list of scenes. It produces model checkpoints and one JSON summary per run.

**Do not run Part 2 during a training loop.** It renders, computes metrics and starts a streaming server, which compete for the GPU and would corrupt the training-time and peak-VRAM measurements.

---

## 0. User configuration

Everything that changes between runs is set in cell 0.1; no other cell needs editing.

**Storage.** `STORAGE_MODE = "drive"` writes the benchmark JSON (and, with `KEEP_MODEL_ON_DRIVE = True`, the model) to Google Drive, so a run survives a runtime restart and the loop can resume. `"local"` keeps everything in `/content`, which is lost when the session ends.

**What to run.** `RUN_MODE = "single"` trains `SCENE`; `"loop"` trains every scene in `SCENES_TO_RUN`. Completed runs are skipped, so re-running a loop after a disconnection resumes from the first missing scene.

**Which protocol.**

| | Protocol A, `"iterations"` | Protocol B, `"target_eval_loss"` |
|---|---|---|
| Stopping criterion | fixed budget `MAX_ITERATIONS` | per-scene target in `TARGET_EVAL_LOSS_PER_SCENE` |
| Metric sampling | every `EVAL_EVERY_N_ITERS` iterations | every `EVAL_EVERY_N_MINUTES` minutes |
| Question answered | at equal budget, which method reconstructs best? | to reach the same quality, what does each method cost? |

**Workflow.** The Protocol B target must be reachable by every method, so it is calibrated from Protocol A: run the Protocol A loop on all scenes in all three notebooks, read the candidates printed by cell 4.3, take the per-scene maximum across the three methods and use the same dictionary in all three notebooks. The calibrated targets used in this study are already filled in.

**Uniform conventions.** All methods are evaluated at 800×800 on the full 20-view test split, on a black background, with LPIPS computed by the VGG backbone. Black is imposed by fudan-zvg: `scene/cameras.py` premultiplies the ground truth by the alpha channel, so its ground truth is black whatever the flag says, and rendering on white against it makes training collapse.

In [ ]:
# 0.1  User configuration: edit this cell, then run Part 1 top to bottom.

# Output storage: "drive" is persistent (survives restarts), "local" lives in /content.
STORAGE_MODE = "drive"

DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/DeformableGaussians_output"
LOCAL_OUTPUT_ROOT = "/content/DeformableGaussians_output"

# "single" trains SCENE only; "loop" trains every scene in SCENES_TO_RUN, in order.
RUN_MODE = "loop"

SCENE = "bouncingballs"

# Used only when RUN_MODE == "loop"; trim the list for a partial loop.
SCENES_TO_RUN = ["bouncingballs", "hellwarrior", "hook", "jumpingjacks",
                 "lego", "mutant", "standup", "trex"]
SCENES_NOT_TO_RUN = []

# Skip runs that are already complete (this makes the loop resumable).
SKIP_TRAINING_IF_TRAINED = True

# False: the model stays in /content and only the benchmark JSON reaches Drive
#        (model storage is measured before the local copy is discarded).
# True:  the model is also copied to Drive, so Part 2 can run later without retraining.
KEEP_MODEL_ON_DRIVE = False
LOCAL_TRAIN_ROOT = "/content/train_outputs"

# Benchmarking protocol: "iterations" (Protocol A) or "target_eval_loss" (Protocol B).
TRAINING_MODE = "iterations"

# Protocol A: fixed iteration budget
MAX_ITERATIONS     = 30000   # optimisation budget
EVAL_EVERY_N_ITERS = 1000    # sample the metrics every N iterations

# Protocol B: fixed quality target
# "eval_loss" stops on the per-scene L1 target below, "psnr" on TARGET_PSNR.
TARGET_METRIC = "eval_loss"
TARGET_PSNR   = 30.0    # used only when TARGET_METRIC == "psnr"

# Per-scene L1 target, identical in the three notebooks and calibrated from the Protocol A
# results of all three methods (cell 4.3, docs/PROTOCOL_B_CALIBRATION.md).
# A scene left at None is skipped by the Protocol B loop.
TARGET_EVAL_LOSS_PER_SCENE = {
    # target = 1.05 x max over methods of the minimum eval L1 on the Protocol A test curve.
    # lego: fudan-zvg is excluded (its training aborts on that scene), so Wu et al. set the target.
    "bouncingballs": 0.005083,   # fudan  0.004841 @18000  x1.05
    "hellwarrior":   0.005358,   # fudan  0.005103 @10000  x1.05
    "hook":          0.006253,   # fudan  0.005956 @4000   x1.05
    "jumpingjacks":  0.004610,   # fudan  0.004391 @4000   x1.05  (partial run, 15000 it)
    "lego":          0.013466,   # Wu     0.012825 @10000  x1.05  (fudan excluded)
    "mutant":        0.003106,   # fudan  0.002958 @11000  x1.05
    "standup":       0.002168,   # fudan  0.002065 @10000  x1.05
    "trex":          0.005793,   # fudan  0.005517 @4000   x1.05  (partial run, 6000 it)
}

EVAL_EVERY_N_MINUTES  = 0.5     # sample the metrics every T minutes
SAFETY_MAX_ITERATIONS = 60000   # hard cap, so a run can never last forever
# The target is ignored below this iteration, so the baseline sample cannot stop the run.
MIN_ITERATIONS_BEFORE_STOP = 1000
# The target must hold for this many consecutive samples (test metrics fluctuate).
TARGET_CONSECUTIVE_HITS = 2

# Evaluation settings (shared by both protocols)
# "l1": mean L1 on the test split; "l1_dssim": (1-w)*L1 + w*(1-SSIM). Both are always recorded;
# this selects the one mirrored by "eval_loss" and used by the Protocol B stop.
EVAL_LOSS_KIND = "l1"
LAMBDA_DSSIM   = 0.2      # weight w of the "l1_dssim" variant

LPIPS_NET      = "vgg"    # LPIPS backbone, as reported in all three papers
MAX_EVAL_VIEWS = 0        # 0 = every test view; must stay 0 for the shared L1 target
EVAL_AT_FIRST_ITERATION = True   # record a baseline point at the first iteration

# Extra flags appended verbatim to the training command (usually empty)
EXTRA_TRAIN_ARGS = ""

# Views consumed per iteration
BATCH_SIZE = 1

METHOD_NAME = "Deformable-3D-Gaussians (Yang et al.)"
REPO_DIR = "/content/Deformable-3D-Gaussians"
D_NERF_SCENES = ["bouncingballs", "hellwarrior", "hook", "jumpingjacks",
                 "lego", "mutant", "standup", "trex"]

assert STORAGE_MODE in ("drive", "local")
assert RUN_MODE in ("single", "loop")
assert TRAINING_MODE in ("iterations", "target_eval_loss")
assert EVAL_LOSS_KIND in ("l1", "l1_dssim")
assert TARGET_METRIC in ("psnr", "eval_loss")
assert SCENE in D_NERF_SCENES, "Unknown scene: %s" % SCENE
assert all(s in D_NERF_SCENES for s in SCENES_TO_RUN)
print("Configuration accepted.")
print("Method   : %s" % METHOD_NAME)
print("Run mode : %s" % RUN_MODE)
print("Protocol : %s" % TRAINING_MODE)
print("Scenes   : %s" % (SCENE if RUN_MODE == "single" else ", ".join(SCENES_TO_RUN)))

In [ ]:
# 0.2  Path resolution: every path is a function of (scene, protocol); Drive is mounted only if requested.
import glob
import json
import os

if STORAGE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = DRIVE_OUTPUT_ROOT
else:
    OUTPUT_ROOT = LOCAL_OUTPUT_ROOT

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(LOCAL_TRAIN_ROOT, exist_ok=True)
TRAINED_GLOB = ("point_cloud", "iteration_*", "point_cloud.ply")


def run_name(scene, mode):
    """Folder name for one (scene, protocol) pair. Runs never collide."""
    if mode == "iterations":
        return "%s_iters%d" % (scene, MAX_ITERATIONS)
    target = TARGET_EVAL_LOSS_PER_SCENE.get(scene)
    tag = ("%g" % target).replace(".", "p") if target is not None else "NA"
    return "%s_loss%s" % (scene, tag)


def paths_for(scene, mode):
    """All output paths for one run.

    The benchmark JSON always lives on OUTPUT_ROOT (Drive when selected), while
    the model goes to Drive only if KEEP_MODEL_ON_DRIVE, otherwise to local
    disk. Metrics are what the benchmark needs; the checkpoints are optional.
    """
    name = run_name(scene, mode)
    model_root = OUTPUT_ROOT if KEEP_MODEL_ON_DRIVE else LOCAL_TRAIN_ROOT
    model_path = os.path.join(model_root, name)
    bench_dir = os.path.join(OUTPUT_ROOT, name, "benchmark")
    return {
        "scene": scene,
        "mode": mode,
        "run_name": name,
        "model_path": model_path,
        "drive_model_path": os.path.join(OUTPUT_ROOT, name),
        "benchmark_dir": bench_dir,
        "bench_json": os.path.join(bench_dir, "benchmark_%s.json" % mode),
        "bench_config": os.path.join(bench_dir, "benchmark_config.json"),
        "source_path": os.path.join(REPO_DIR, "dataset_dir", "data", scene),
    }


def is_model_trained(model_path):
    """True when the model files of that run are present on disk."""
    return len(glob.glob(os.path.join(model_path, *TRAINED_GLOB))) > 0


TERMINAL_STATUS = ("finished", "stopped_on_target", "stopped_on_safety_cap")


def run_is_complete(p):
    """True when this run has already produced a finished benchmark JSON.

    Completion is judged on the JSON rather than on the model, because with
    KEEP_MODEL_ON_DRIVE = False the model does not survive the runtime. This is
    what makes the loop resumable after a disconnection or a GPU-quota stop.
    """
    if not os.path.exists(p["bench_json"]):
        return False
    try:
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
    except Exception:
        return False
    if summary.get("status") not in TERMINAL_STATUS:
        return False
    if KEEP_MODEL_ON_DRIVE and not is_model_trained(p["drive_model_path"]):
        return False
    return True


def activate_scene(scene, mode=None):
    """Point the module-level variables at one scene. Part 2 reads these."""
    global SCENE, SOURCE_PATH, MODEL_OUTPUT_PATH, MODEL_DIR
    global BENCHMARK_DIR, BENCH_JSON_PATH, BENCH_CONFIG_PATH, RUN_NAME
    mode = mode or TRAINING_MODE
    p = paths_for(scene, mode)
    SCENE = scene
    SOURCE_PATH = p["source_path"]
    MODEL_OUTPUT_PATH = p["model_path"]
    MODEL_DIR = p["model_path"]          # alias used by Part 2
    BENCHMARK_DIR = p["benchmark_dir"]
    BENCH_JSON_PATH = p["bench_json"]
    BENCH_CONFIG_PATH = p["bench_config"]
    RUN_NAME = run_name(scene, mode)
    os.makedirs(BENCHMARK_DIR, exist_ok=True)
    return p


activate_scene(SCENE)

print("Metrics root : %s" % OUTPUT_ROOT)
print("Model folder : %s%s" % (MODEL_OUTPUT_PATH,
                               "" if KEEP_MODEL_ON_DRIVE else "   (not kept)"))
print("Benchmark    : %s" % BENCH_JSON_PATH)
print("Already done : %s" % run_is_complete(paths_for(SCENE, TRAINING_MODE)))

## 1. Environment and repository setup

In [ ]:
# Check the GPU assigned by Colab (e.g. Tesla T4)
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/ingra14m/Deformable-3D-Gaussians --recursive
%cd Deformable-3D-Gaussians

In [ ]:
%env TORCH_CUDA_ARCH_LIST=7.5
%env MAX_JOBS=4

!pip install -q plyfile

# Build the CUDA submodules against Colab's default PyTorch
!pip install ./submodules/depth-diff-gaussian-rasterization
!pip install ./submodules/simple-knn

## 2. CUDA module fixes

`simple-knn` needs `<float.h>` for its floating-point limits; the module is then reinstalled with its own `setup.py`.

In [ ]:
import os

os.chdir('/content/Deformable-3D-Gaussians/submodules/simple-knn')

filepath = 'simple_knn.cu'
with open(filepath, 'r') as f:
    content = f.read()

if '#include <float.h>' not in content:
    with open(filepath, 'w') as f:
        f.write('#include <float.h>\n' + content)
    print('Patch applied successfully.')
else:
    print('Patch already present.')

!python setup.py install

os.chdir('/content/Deformable-3D-Gaussians')
print("Back in the main directory.")

## 3. Dataset download

The D-NeRF synthetic dataset (all eight scenes) is downloaded once per runtime; the scene is selected per run.

In [ ]:
# Download the D-NeRF dataset once per runtime (the archive contains all eight scenes).
import os

os.chdir(REPO_DIR)

if not os.path.isdir("dataset_dir/data"):
    !mkdir -p dataset_dir
    !wget -q "https://www.dropbox.com/scl/fi/cdcmkufncwcikk1dzbgb4/data.zip?rlkey=n5m21i84v2b2xk6h7qgiu8nkg&dl=1" -O data.zip
    !unzip -q data.zip -d dataset_dir/
else:
    print("Dataset already present in this runtime, download skipped.")

DATASET_ROOT = os.path.join(REPO_DIR, "dataset_dir", "data")
available = sorted(d for d in os.listdir(DATASET_ROOT)
                   if os.path.isdir(os.path.join(DATASET_ROOT, d)))
print("Scenes available: %s" % available)

missing = [s for s in D_NERF_SCENES if s not in available]
assert not missing, "Scenes missing from the archive: %s" % missing

activate_scene(SCENE)
print("Active scene: %s -> %s" % (SCENE, SOURCE_PATH))

## 3.5 Per-scene preparation

Deformable-3DGS is configured entirely through command-line flags, so no per-scene file is needed. The functions defined here form the interface the training driver uses for every method: `prepare_scene` returns the extra flags for a scene, `build_train_command` assembles the command and `model_storage_report` measures the files needed to render the model.

The background is left at the repository default (black), which matches the D-NeRF setting of the paper and the convention imposed by fudan-zvg.

In [ ]:
# 3.5  Per-scene preparation and training command (Deformable-3DGS: command-line flags only)
import glob
import os


def prepare_scene(scene, mode):
    """Nothing to prepare; returns the extra CLI flags for this scene."""
    return EXTRA_TRAIN_ARGS


def build_train_command(p, budget, extra):
    cmd = ('python train.py'
           + ' -s "%s"' % p["source_path"]
           + ' -m "%s"' % p["model_path"]
           + ' --eval --is_blender'
           + ' --iterations %d' % budget
           + ' --test_iterations 0'
           + ' --save_iterations %d' % budget)
    return cmd + ((' ' + extra) if extra else '')


def _folder_size_mb(root):
    total = 0
    for dirpath, _d, filenames in os.walk(root):
        for fn in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, fn))
            except OSError:
                pass
    return total / 1048576.0


def _sum_patterns(model_path, patterns):
    files = []
    for pat in patterns:
        files += sorted(glob.glob(os.path.join(model_path, pat)))
    files = [f for f in files if os.path.isfile(f)]
    total = sum(os.path.getsize(f) for f in files)
    return files, total


# Files needed to render and evaluate the model (verified by leave-one-out).
REQUIRED_PATTERNS = ("cfg_args",
                     "point_cloud/iteration_*/point_cloud.ply",
                     "deform/iteration_*/deform.pth")


def model_storage_report(model_path):
    files, total = _sum_patterns(model_path, REQUIRED_PATTERNS)
    return {
        "required_files": [os.path.relpath(f, model_path) for f in files],
        "model_storage_mb": round(total / 1048576.0, 4),
        "full_folder_mb": round(_folder_size_mb(model_path), 4),
    }


print("Scene preparation ready (no per-scene files needed).")

## 4. Training with incremental benchmarking

**Instrumentation.** Each repository's training loop calls its own `training_report(...)` at every iteration. Instead of rewriting the loop, which would alter densification schedules, learning-rate ramps and stage handling, the notebook appends a few lines to `train.py` that wrap that function with a monitor. The upstream code path, hyper-parameters and command line stay untouched, a pristine copy is kept as `train.py.orig`, and without the `BENCH_CONFIG` environment variable `train.py` behaves exactly as upstream. The wrapper recovers the loop state through `inspect`, so it is robust to signature differences.

At every sampling point the monitor evaluates the current model on the full test split with the repository's own modules (`utils.image_utils.psnr`, `utils.loss_utils.ssim`, `utils.loss_utils.l1_loss`, bundled `lpipsPyTorch` with VGG) and records the metrics, the training time net of the benchmark overhead, the iteration count, the images seen, the number of Gaussians and the evaluation resolution. The JSON summary is rewritten atomically after each evaluation, so an interrupted run still leaves a valid file. Native periodic testing is disabled (`--test_iterations 0`) so that it does not add to the measured time.

**Protocol B cost.** A run stops only after the target has held for `TARGET_CONSECUTIVE_HITS` consecutive samples, so the termination time overstates the cost, by a different amount for each method. The figure to report is the **first crossing**, extracted by cell 4.4.

In [ ]:
# 4.0  Install the benchmark monitor and build the training driver
import json
import os
import shutil
import subprocess
import sys
import threading
import time

os.chdir(REPO_DIR)

MONITOR_SRC = r'''# benchmark_monitor.py -- generated by the notebook: incremental benchmark logger for __METHOD_NAME__.
# Wraps the repository training_report(...), which the training loop calls at every iteration,
# evaluates the current model on the test split with the repository metric modules and appends
# one record per sample to a JSON summary. In Protocol B it saves the model and stops on target.

import atexit
import inspect
import json
import os
import sys
import time

import torch

from utils.image_utils import psnr as _repo_psnr
from utils.loss_utils import ssim as _repo_ssim
from utils.loss_utils import l1_loss as _repo_l1


# Configuration written by the notebook, passed through $BENCH_CONFIG

_CFG_PATH = os.environ.get("BENCH_CONFIG", "")

with open(_CFG_PATH, "r") as _f:
    CFG = json.load(_f)

METHOD = CFG["method"]
SCENE = CFG["scene"]
MODE = CFG["mode"]                                   # "iterations" | "target_eval_loss"
JSON_PATH = CFG["json_path"]

EVAL_EVERY_N_ITERS = int(CFG.get("eval_every_n_iters", 2000))
EVAL_EVERY_N_SECONDS = float(CFG.get("eval_every_n_seconds", 600.0))
MAX_ITERATIONS = int(CFG.get("max_iterations", 30000))
SAFETY_MAX_ITERATIONS = int(CFG.get("safety_max_iterations", 100000))

# Protocol B stopping criterion: "eval_loss" (eval_loss <= target) or "psnr" (psnr >= target)
TARGET_METRIC = CFG.get("target_metric", "psnr")
TARGET_PSNR = float(CFG.get("target_psnr", 30.0))
TARGET_EVAL_LOSS = float(CFG.get("target_eval_loss", 0.0))
MIN_ITERATIONS_BEFORE_STOP = int(CFG.get("min_iterations_before_stop", 2000))
# Test metrics are not monotonic: the target must hold for several consecutive samples.
TARGET_CONSECUTIVE_HITS = int(CFG.get("target_consecutive_hits", 2))

# Evaluation loss: a common photometric yardstick, not any method's own objective
# (regularisers never look at the ground truth and have no evaluation counterpart).
EVAL_LOSS_KIND = CFG.get("eval_loss_kind", "l1")     # "l1" | "l1_dssim"
LAMBDA_DSSIM = float(CFG.get("lambda_dssim", 0.2))
LPIPS_NET = CFG.get("lpips_net", "vgg")
MAX_EVAL_VIEWS = int(CFG.get("max_eval_views", 0))   # 0 = all test views
EVAL_AT_FIRST_ITERATION = bool(CFG.get("eval_at_first_iteration", True))

# BATCH_SIZE: views per iteration (per-scene batch for fudan-zvg, 1 for the other two).
# ITERATION_OFFSET: iterations of an earlier stage whose counter restarted (coarse stage of Wu et al.).
BATCH_SIZE = int(CFG.get("batch_size", 1))
ITERATION_OFFSET = int(CFG.get("iteration_offset", 0))


# LPIPS: bundled lpipsPyTorch, falling back to the pip package; the model is built once.

_lpips_model = None
_lpips_backend = None


def _get_lpips():
    global _lpips_model, _lpips_backend
    if _lpips_model is not None:
        return _lpips_model, _lpips_backend
    try:
        from lpipsPyTorch.modules.lpips import LPIPS
        _lpips_model = LPIPS(net_type=LPIPS_NET).to("cuda").eval()
        _lpips_backend = "lpipsPyTorch"
    except Exception as exc:
        print("[benchmark] lpipsPyTorch unavailable (%r), falling back to the "
              "lpips pip package." % (exc,))
        import lpips as _lpips_pkg
        _lpips_model = _lpips_pkg.LPIPS(net=LPIPS_NET).to("cuda").eval()
        _lpips_backend = "lpips-pip"
    return _lpips_model, _lpips_backend


def _lpips_value(image, gt):
    model, backend = _get_lpips()
    x = image.unsqueeze(0)
    y = gt.unsqueeze(0)
    if backend == "lpips-pip":
        # the pip package expects inputs in [-1, 1]
        x = x * 2.0 - 1.0
        y = y * 2.0 - 1.0
    return model(x, y).mean().item()


# JSON summary, rewritten atomically at every evaluation

_SUMMARY = {
    "method": METHOD,
    "scene": SCENE,
    "mode": MODE,
    "config": CFG,
    "status": "running",
    "target_reached": False,
    "entries": [],
    "final": None,
}


def _flush_summary():
    directory = os.path.dirname(JSON_PATH)
    if directory:
        os.makedirs(directory, exist_ok=True)
    tmp_path = JSON_PATH + ".tmp"
    with open(tmp_path, "w") as handle:
        json.dump(_SUMMARY, handle, indent=2)
    os.replace(tmp_path, JSON_PATH)


@atexit.register
def _on_exit():
    if _SUMMARY["status"] == "running":
        _SUMMARY["status"] = "finished"
    if _SUMMARY["entries"]:
        _SUMMARY["final"] = _SUMMARY["entries"][-1]
    try:
        _flush_summary()
    except Exception as exc:
        print("[benchmark] could not write the JSON summary: %r" % (exc,))


class _Monitor(object):

    def __init__(self):
        self.t_start = None
        self.eval_overhead_s = 0.0
        self.last_eval_time = None
        self.last_eval_iteration = -1
        self.best_psnr = -1.0
        self.n_evals = 0
        self.consecutive_hits = 0

    def _target_met(self, entry):
        if entry["iteration"] < MIN_ITERATIONS_BEFORE_STOP:
            return False
        if TARGET_METRIC == "psnr":
            return entry["psnr"] >= TARGET_PSNR
        return entry["eval_loss"] <= TARGET_EVAL_LOSS

    def _must_evaluate(self, iteration, now):
        if iteration == self.last_eval_iteration:
            return False
        if EVAL_AT_FIRST_ITERATION and self.n_evals == 0:
            return True
        if MODE == "iterations":
            if iteration >= MAX_ITERATIONS:
                return True
            return (iteration % EVAL_EVERY_N_ITERS) == 0
        # MODE == "target_eval_loss": time-driven sampling
        if iteration >= SAFETY_MAX_ITERATIONS:
            return True
        return (now - self.last_eval_time) >= EVAL_EVERY_N_SECONDS

    @torch.no_grad()
    def _evaluate(self, a):
        psnr_sum, ssim_sum, lpips_sum, l1_sum, n_views = 0.0, 0.0, 0.0, 0.0, 0
        eval_h, eval_w = 0, 0
        for view in _iter_eval_views(a):
            image, gt = _render_pair(a, view)
            eval_h, eval_w = int(image.shape[-2]), int(image.shape[-1])
            image_b = image.unsqueeze(0)
            gt_b = gt.unsqueeze(0)
            psnr_sum += _repo_psnr(image_b, gt_b).mean().item()
            ssim_sum += _repo_ssim(image, gt).mean().item()
            l1_sum += _repo_l1(image, gt).mean().item()
            lpips_sum += _lpips_value(image, gt)
            n_views += 1
        n_views = max(n_views, 1)
        return {
            "psnr": psnr_sum / n_views,
            "ssim": ssim_sum / n_views,
            "lpips": lpips_sum / n_views,
            "eval_l1_loss": l1_sum / n_views,
            "num_eval_views": n_views,
            "eval_resolution": [eval_h, eval_w],
        }

    def step(self, a):
        now = time.time()
        if self.t_start is None:
            self.t_start = now
            self.last_eval_time = now

        iteration = int(a["iteration"])
        stage = a.get("stage", None)

        if not _stage_is_evaluable(stage):
            return
        if not self._must_evaluate(iteration, now):
            return

        eval_t0 = time.time()
        torch.cuda.empty_cache()
        metrics = self._evaluate(a)
        torch.cuda.synchronize()
        eval_dt = time.time() - eval_t0

        # Both loss variants are always recorded (SSIM is computed anyway)
        eval_photometric = ((1.0 - LAMBDA_DSSIM) * metrics["eval_l1_loss"]
                            + LAMBDA_DSSIM * (1.0 - metrics["ssim"]))
        eval_loss = (eval_photometric if EVAL_LOSS_KIND == "l1_dssim"
                     else metrics["eval_l1_loss"])

        wall = eval_t0 - self.t_start
        entry = {
            "iteration": iteration,
            "total_iterations": iteration + ITERATION_OFFSET,
            "images_seen": (iteration + ITERATION_OFFSET) * BATCH_SIZE,
            "stage": stage if stage is not None else "single_stage",
            "training_time_s": wall - self.eval_overhead_s,
            "wall_time_s": wall,
            "benchmark_overhead_s": self.eval_overhead_s,
            "psnr": metrics["psnr"],
            "ssim": metrics["ssim"],
            "lpips": metrics["lpips"],
            "eval_l1_loss": metrics["eval_l1_loss"],
            "eval_photometric_loss": eval_photometric,
            "eval_loss": eval_loss,
            "eval_loss_kind": EVAL_LOSS_KIND,
            "num_eval_views": metrics["num_eval_views"],
            "eval_resolution": metrics["eval_resolution"],
            "num_gaussians": int(a["scene"].gaussians.get_xyz.shape[0]),
            "train_batch_loss": float(a["loss"].item()) if hasattr(a.get("loss", None), "item") else None,
            "eval_duration_s": eval_dt,
        }

        self.last_eval_iteration = iteration
        self.n_evals += 1

        _SUMMARY["entries"].append(entry)
        _SUMMARY["final"] = entry
        _flush_summary()

        print("\n[benchmark] iter %7d | train %8.1fs | PSNR %6.3f | SSIM %.4f | "
              "LPIPS %.4f | eval_loss %.5f | #G %d | eval@%dx%d"
              % (iteration, entry["training_time_s"], entry["psnr"], entry["ssim"],
                 entry["lpips"], entry["eval_loss"], entry["num_gaussians"],
                 entry["eval_resolution"][1], entry["eval_resolution"][0]),
              flush=True)

        _maybe_track_best(a, entry, self)

        # Evaluation, JSON flush and best-checkpoint writes are benchmark overhead, not training time.
        self.eval_overhead_s += time.time() - eval_t0
        self.last_eval_time = time.time()

        if MODE == "target_eval_loss":
            if self._target_met(entry):
                self.consecutive_hits += 1
                if TARGET_METRIC == "psnr":
                    print("[benchmark] target met (%d/%d consecutive): PSNR %.4f >= %.4f"
                          % (self.consecutive_hits, TARGET_CONSECUTIVE_HITS,
                             entry["psnr"], TARGET_PSNR), flush=True)
                else:
                    print("[benchmark] target met (%d/%d consecutive): eval_loss "
                          "%.6f <= %.6f" % (self.consecutive_hits,
                                            TARGET_CONSECUTIVE_HITS,
                                            eval_loss, TARGET_EVAL_LOSS), flush=True)
            else:
                if self.consecutive_hits:
                    print("[benchmark] target lost again, consecutive counter reset.",
                          flush=True)
                self.consecutive_hits = 0

            if self.consecutive_hits >= TARGET_CONSECUTIVE_HITS:
                criterion = ("PSNR >= %.4f" % TARGET_PSNR if TARGET_METRIC == "psnr"
                             else "eval_loss <= %.6f" % TARGET_EVAL_LOSS)
                print("\n[benchmark] TARGET REACHED (%s held for %d consecutive "
                      "sampling points) at iteration %d. Saving the model and "
                      "stopping training." % (criterion, TARGET_CONSECUTIVE_HITS,
                                              iteration), flush=True)
                _SUMMARY["target_reached"] = True
                _SUMMARY["status"] = "stopped_on_target"
                _SUMMARY["target_metric"] = TARGET_METRIC
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)
            if iteration >= SAFETY_MAX_ITERATIONS:
                print("\n[benchmark] Safety cap of %d iterations reached without "
                      "hitting the target eval loss. Saving and stopping."
                      % SAFETY_MAX_ITERATIONS, flush=True)
                _SUMMARY["status"] = "stopped_on_safety_cap"
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)


_MONITOR = _Monitor()


# Hook factory, called from the block injected at the bottom of train.py

def make_hook(original_training_report):
    signature = inspect.signature(original_training_report)

    def hooked_training_report(*args, **kwargs):
        result = original_training_report(*args, **kwargs)
        try:
            bound = signature.bind(*args, **kwargs)
            bound.apply_defaults()
            _MONITOR.step(bound.arguments)
        except SystemExit:
            raise
        except Exception as exc:
            print("[benchmark] evaluation hook failed: %r" % (exc,), flush=True)
        return result

    print("[benchmark] monitor installed (method=%s, scene=%s, mode=%s)"
          % (METHOD, SCENE, MODE), flush=True)
    return hooked_training_report


# Repository-specific glue: ingra14m/Deformable-3D-Gaussians. A test view is rendered by
# querying the deformation MLP at the view timestamp, as the repository does.

def _stage_is_evaluable(stage):
    return True


def _iter_eval_views(a):
    cameras = a["scene"].getTestCameras()
    total = len(cameras)
    indices = list(range(total))
    if MAX_EVAL_VIEWS and MAX_EVAL_VIEWS < total:
        step = max(1, total // MAX_EVAL_VIEWS)
        indices = indices[::step][:MAX_EVAL_VIEWS]
    for index in indices:
        yield cameras[index]


def _render_pair(a, view):
    load_on_the_fly = bool(a.get("load2gpu_on_the_fly", False))
    if load_on_the_fly:
        view.load2device()

    xyz = a["scene"].gaussians.get_xyz
    fid = view.fid.to("cuda")
    time_input = fid.unsqueeze(0).expand(xyz.shape[0], -1)
    d_xyz, d_rotation, d_scaling = a["deform"].step(xyz.detach(), time_input)

    render_pkg = a["renderFunc"](view, a["scene"].gaussians, *a["renderArgs"],
                                 d_xyz, d_rotation, d_scaling,
                                 bool(a.get("is_6dof", False)))
    image = torch.clamp(render_pkg["render"], 0.0, 1.0)
    gt = torch.clamp(view.original_image.to("cuda"), 0.0, 1.0)[0:3, :, :]

    if load_on_the_fly:
        view.load2device("cpu")
    return image, gt


def _save_model(a):
    iteration = int(a["iteration"])
    scene = a["scene"]
    scene.save(iteration)
    a["deform"].save_weights(scene.model_path, iteration)
    print("[benchmark] Gaussians + deformation network saved at iteration %d"
          % iteration, flush=True)


def _maybe_track_best(a, entry, monitor):
    if entry["psnr"] > monitor.best_psnr:
        monitor.best_psnr = entry["psnr"]
        _SUMMARY["best_psnr"] = entry["psnr"]
        _SUMMARY["best_psnr_iteration"] = entry["iteration"]
'''

with open(os.path.join(REPO_DIR, "benchmark_monitor.py"), "w") as handle:
    handle.write(MONITOR_SRC)

# Inject the hook into train.py (idempotent; a pristine copy is kept as train.py.orig)
TRAIN_PY = os.path.join(REPO_DIR, "train.py")
BACKUP_PY = TRAIN_PY + ".orig"
if not os.path.exists(BACKUP_PY):
    shutil.copyfile(TRAIN_PY, BACKUP_PY)
shutil.copyfile(BACKUP_PY, TRAIN_PY)

with open(TRAIN_PY, "r") as handle:
    source = handle.read()

INJECTED = (
    "# === BENCHMARK HOOK (injected by the notebook) ===\n"
    "import os as _bench_os\n"
    "if _bench_os.environ.get('BENCH_CONFIG'):\n"
    "    import benchmark_monitor as _bench\n"
    "    training_report = _bench.make_hook(training_report)\n"
    "# === END BENCHMARK HOOK ===\n\n"
)
ANCHOR = 'if __name__ == "__main__":'
assert source.count(ANCHOR) == 1, "Unexpected train.py layout."
with open(TRAIN_PY, "w") as handle:
    handle.write(source.replace(ANCHOR, INJECTED + ANCHOR, 1))

print("benchmark_monitor.py written and hook injected into train.py.")
print("Without the BENCH_CONFIG environment variable train.py behaves exactly "
      "as upstream (pristine copy kept as train.py.orig).")


# Peak VRAM: training runs in a subprocess, so nvidia-smi is polled for the whole device.
class VramSampler(threading.Thread):

    def __init__(self, period_s=2.0):
        threading.Thread.__init__(self)
        self.daemon = True
        self.period_s = period_s
        self.peak_mb = 0.0
        self._stop_event = threading.Event()

    def run(self):
        while not self._stop_event.is_set():
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"])
                self.peak_mb = max(self.peak_mb,
                                   float(out.decode().strip().splitlines()[0]))
            except Exception:
                pass
            self._stop_event.wait(self.period_s)

    def stop(self):
        self._stop_event.set()
        return self.peak_mb


def write_bench_config(p):
    """Serialise the benchmark settings read by benchmark_monitor.py."""
    target = TARGET_EVAL_LOSS_PER_SCENE.get(p["scene"])
    config = {
        "method": METHOD_NAME,
        "scene": p["scene"],
        "mode": p["mode"],
        "json_path": p["bench_json"],
        "eval_every_n_iters": int(EVAL_EVERY_N_ITERS),
        "eval_every_n_seconds": float(EVAL_EVERY_N_MINUTES) * 60.0,
        "max_iterations": int(MAX_ITERATIONS),
        "safety_max_iterations": int(SAFETY_MAX_ITERATIONS),
        "target_metric": TARGET_METRIC,
        "target_psnr": float(TARGET_PSNR),
        "target_eval_loss": float(target) if target is not None else 0.0,
        "min_iterations_before_stop": int(MIN_ITERATIONS_BEFORE_STOP),
        "target_consecutive_hits": int(TARGET_CONSECUTIVE_HITS),
        "eval_loss_kind": EVAL_LOSS_KIND,
        "lambda_dssim": float(LAMBDA_DSSIM),
        "lpips_net": LPIPS_NET,
        "max_eval_views": int(MAX_EVAL_VIEWS),
        "eval_at_first_iteration": bool(EVAL_AT_FIRST_ITERATION),
        "source_path": p["source_path"],
        "model_path": p["model_path"],
        "batch_size": int(BATCH_SIZE),
        "iteration_offset": int(globals().get("COARSE_ITERATIONS", 0)),
    }
    os.makedirs(p["benchmark_dir"], exist_ok=True)
    with open(p["bench_config"], "w") as handle:
        json.dump(config, handle, indent=2)
    return config


def record_run_cost(p, wall_s, peak_vram_mb):
    if not os.path.exists(p["bench_json"]):
        return
    with open(p["bench_json"], "r") as handle:
        summary = json.load(handle)
    summary["process_wall_time_s"] = wall_s
    summary["peak_vram_mb_nvidia_smi"] = peak_vram_mb
    with open(p["bench_json"], "w") as handle:
        json.dump(summary, handle, indent=2)


def train_scene(scene, mode=None):
    """Train one scene under one protocol. Returns a short status string.

    The loop needs this to be a function, so the training command is issued
    with subprocess rather than the `!` magic (which cannot appear inside a
    function body). Output is streamed line by line, as `!` would do.
    """
    mode = mode or TRAINING_MODE
    p = activate_scene(scene, mode)

    if mode == "target_eval_loss" and TARGET_METRIC == "eval_loss":
        if TARGET_EVAL_LOSS_PER_SCENE.get(scene) is None:
            print("[%s] no target calibrated for this scene -> skipped." % scene)
            return "skipped_no_target"

    if SKIP_TRAINING_IF_TRAINED and run_is_complete(p):
        print("[%s] already benchmarked -> skipped (%s)."
              % (scene, p["bench_json"]))
        return "skipped_already_trained"

    os.chdir(REPO_DIR)
    extra = prepare_scene(scene, mode)
    write_bench_config(p)
    budget = MAX_ITERATIONS if mode == "iterations" else SAFETY_MAX_ITERATIONS
    cmd = build_train_command(p, budget, extra)

    print("=" * 78)
    print("[%s | %s] %s" % (scene, mode, cmd))
    print("=" * 78, flush=True)

    env = dict(os.environ, BENCH_CONFIG=p["bench_config"])
    sampler = VramSampler()
    sampler.start()
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    wall = time.time() - t0
    peak = sampler.stop()
    record_run_cost(p, wall, peak)

    # Measure model storage now, before the local model is discarded.
    try:
        storage = model_storage_report(p["model_path"])
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
        summary["storage"] = storage
        with open(p["bench_json"], "w") as handle:
            json.dump(summary, handle, indent=2)
        print("[%s] model storage %.2f MB (full folder %.2f MB)"
              % (scene, storage["model_storage_mb"], storage["full_folder_mb"]))
    except Exception as exc:
        print("[%s] storage measurement failed: %r" % (scene, exc))

    if KEEP_MODEL_ON_DRIVE:
        print("[%s] model kept at %s" % (scene, p["model_path"]))
    else:
        shutil.rmtree(p["model_path"], ignore_errors=True)
        print("[%s] local model discarded; metrics kept on Drive." % scene)


    print("\n[%s] finished in %dm %ds (return code %d), peak VRAM %.0f MB"
          % (scene, int(wall // 60), int(wall % 60), proc.returncode, peak),
          flush=True)
    return "ok" if proc.returncode == 0 else "failed(%d)" % proc.returncode


print("Training driver ready.")

### 4.1 Single-scene training

Trains `SCENE` under the protocol selected by `TRAINING_MODE`. Inert unless `RUN_MODE == "single"`.

In [ ]:
# 4.1  Train a single scene (inert unless RUN_MODE == "single")
if RUN_MODE != "single":
    print("RUN_MODE is 'loop': use cell 4.2 instead.")
else:
    status = train_scene(SCENE)
    print("Status: %s" % status)

### 4.2 Training loop

Trains every scene in `SCENES_TO_RUN` sequentially, one run fully written to storage before the next starts. Inert unless `RUN_MODE == "loop"`. The loop is resumable: completed runs are skipped, so re-running the cell after a disconnection or a GPU-quota stop continues where it stopped. A scene that raises an exception is recorded and the loop moves on. Scenes run one at a time on purpose, since sharing the GPU would corrupt the time and VRAM measurements.

In [ ]:
# 4.2  Train every scene in SCENES_TO_RUN (re-run the cell to resume after a disconnection)
import time

if RUN_MODE != "loop":
    print("RUN_MODE is 'single': use cell 4.1 instead.")
else:
    loop_results = {}
    loop_t0 = time.time()
    for _i, _scene in enumerate(SCENES_TO_RUN, 1):
        print("\n\n### [%d/%d] %s ###" % (_i, len(SCENES_TO_RUN), _scene),
              flush=True)
        try:
            loop_results[_scene] = train_scene(_scene)
        except KeyboardInterrupt:
            loop_results[_scene] = "interrupted"
            break
        except Exception as _exc:
            loop_results[_scene] = "error: %r" % (_exc,)
            print("[%s] ERROR: %r -- continuing with the next scene."
                  % (_scene, _exc), flush=True)

    print("\n" + "=" * 60)
    print(" LOOP SUMMARY (%s, %.1f min)"
          % (TRAINING_MODE, (time.time() - loop_t0) / 60.0))
    print("=" * 60)
    for _scene in SCENES_TO_RUN:
        print("  %-16s %s" % (_scene, loop_results.get(_scene, "not reached")))

### 4.3 Protocol B target calibration

Run after the Protocol A loop. Prints this method's final eval L1 per scene and the candidate target with a 5% margin. The target used in `TARGET_EVAL_LOSS_PER_SCENE` is the **largest candidate across the three methods**, identical in all three notebooks; the margin keeps the slowest method from meeting the target only where its curve flattens.

Note: the targets used in this study apply the same rule to the **minimum** of each Protocol A test curve rather than to the final sample. The two agree within 2% for Deformable-3DGS and 4DGaussians; for fudan-zvg the final sample is degraded by post-peak overfitting (see `docs/PROTOCOL_B_CALIBRATION.md`).

In [ ]:
# 4.3  Protocol B calibration: prints this method's candidate targets only. Take the per-scene
# maximum over the three methods and paste it into all three notebooks.
import json
import os

print("%-16s %14s %14s" % ("scene", "final eval L1", "candidate x1.05"))
print("-" * 48)
candidates = {}
for _scene in D_NERF_SCENES:
    _p = paths_for(_scene, "iterations")
    if not os.path.exists(_p["bench_json"]):
        print("%-16s %14s %14s" % (_scene, "-", "-"))
        continue
    with open(_p["bench_json"], "r") as handle:
        _summary = json.load(handle)
    _final = _summary.get("final")
    if not _final:
        print("%-16s %14s %14s" % (_scene, "-", "-"))
        continue
    _l1 = _final["eval_l1_loss"]
    candidates[_scene] = round(_l1 * 1.05, 6)
    print("%-16s %14.6f %14.6f" % (_scene, _l1, candidates[_scene]))

print("\nThis method's candidates (compare with the other two, keep the max):")
print(json.dumps(candidates, indent=4))

### 4.4 Benchmark summary

Reads the JSON of the active scene. After a loop, call `activate_scene("<scene>")` to inspect another one.

In [ ]:
# 4.4  Benchmark summary for the active scene
import json
import os

import pandas as pd

train_duration = None
benchmark_summary = None
benchmark_table = None

if os.path.exists(BENCH_JSON_PATH):
    with open(BENCH_JSON_PATH, "r") as handle:
        benchmark_summary = json.load(handle)
    entries = benchmark_summary.get("entries", [])
else:
    entries = []

if entries:
    _all = pd.DataFrame(entries)
    columns = [c for c in ["iteration", "total_iterations", "images_seen",
                           "training_time_s", "psnr", "ssim", "lpips",
                           "eval_loss", "eval_l1_loss", "eval_photometric_loss",
                           "num_gaussians", "stage"] if c in _all.columns]
    benchmark_table = _all[columns]
    pd.set_option("display.float_format", lambda v: "%.5f" % v)

    print("Method   : %s" % benchmark_summary["method"])
    print("Scene    : %s" % benchmark_summary["scene"])
    print("Protocol : %s" % benchmark_summary["mode"])
    print("Status   : %s" % benchmark_summary["status"])

    final = benchmark_summary["final"]
    train_duration = final["training_time_s"]

    if benchmark_summary["mode"] == "target_eval_loss":
        _cfg = benchmark_summary.get("config", {})
        _target = _cfg.get("target_eval_loss")
        print("Target   : eval_loss <= %.6f, held for %d consecutive samples"
              % (_target, _cfg.get("target_consecutive_hits", 2)))
        print("Reached  : %s" % benchmark_summary.get("target_reached"))
        # Report the first crossing of the target, not the termination (which includes the hysteresis).
        _first = next((e for e in entries if e["eval_loss"] <= _target), None)
        if _first is not None:
            print("\n  First crossing  : iteration %d, %.1f s (%.2f min)"
                  % (_first["iteration"], _first["training_time_s"],
                     _first["training_time_s"] / 60.0))
            print("  Stop confirmed  : iteration %d, %.1f s (%.2f min)"
                  % (final["iteration"], train_duration, train_duration / 60.0))
            print("  Hysteresis delay: %.1f s (+%.0f%%)"
                  % (train_duration - _first["training_time_s"],
                     100.0 * (train_duration / max(_first["training_time_s"], 1e-9) - 1.0)))
            print("  --> report the FIRST CROSSING as the cost of reaching the target.")
    print()
    display(benchmark_table)

    print()
    print("=" * 64)
    print(" FINAL POINT OF THE RUN")
    print("=" * 64)
    print("  Iterations                : %d" % final["iteration"])
    print("  Training time (net)       : %.2f s (%.2f min)"
          % (train_duration, train_duration / 60.0))
    print("  Benchmark overhead        : %.2f s" % final["benchmark_overhead_s"])
    print("  PSNR                      : %.4f dB" % final["psnr"])
    print("  SSIM                      : %.4f" % final["ssim"])
    print("  LPIPS                     : %.4f" % final["lpips"])
    print("  Eval loss (%-8s)      : %.6f" % (final["eval_loss_kind"], final["eval_loss"]))
    print("  Number of Gaussians       : %d" % final["num_gaussians"])
    if "eval_resolution" in final:
        print("  Evaluation resolution     : %d x %d  <-- must match across methods"
              % (final["eval_resolution"][1], final["eval_resolution"][0]))
    if "peak_vram_mb_nvidia_smi" in benchmark_summary:
        print("  Peak VRAM (device)        : %.0f MB"
              % benchmark_summary["peak_vram_mb_nvidia_smi"])
    print("=" * 64)
    print("JSON: %s" % BENCH_JSON_PATH)
else:
    print("No benchmark data for the active scene (%s, %s)." % (SCENE, TRAINING_MODE))

if benchmark_table is not None and len(benchmark_table) > 1:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    minutes = benchmark_table["training_time_s"] / 60.0
    axes[0].plot(minutes, benchmark_table["psnr"], marker="o", color="tab:blue")
    axes[0].set_xlabel("training time [min]"); axes[0].set_ylabel("PSNR [dB]")
    axes[0].set_title("PSNR (higher is better)")
    axes[1].plot(minutes, benchmark_table["ssim"], marker="o", color="tab:green")
    axes[1].set_xlabel("training time [min]"); axes[1].set_ylabel("SSIM")
    axes[1].set_title("SSIM (higher is better)")
    axes[2].plot(minutes, benchmark_table["lpips"], marker="s", color="tab:red")
    axes[2].set_xlabel("training time [min]"); axes[2].set_ylabel("LPIPS")
    axes[2].set_title("LPIPS (lower is better)")
    axes[3].plot(benchmark_table["iteration"], benchmark_table["eval_loss"],
                 marker="o", color="tab:orange")
    axes[3].set_xlabel("iteration"); axes[3].set_ylabel("evaluation loss")
    axes[3].set_title("Evaluation loss (lower is better)")
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.suptitle("%s -- %s (%s)" % (METHOD_NAME, SCENE, TRAINING_MODE))
    plt.tight_layout()
    plt.show()

---

# Part 2: evaluation and visualisation

**Do not run this part while a training loop is in progress.** Run cells 0.1 and 0.2 first (and the environment cells after a runtime restart), then call `activate_scene("<scene>")` to select the trained model. With `KEEP_MODEL_ON_DRIVE = False` the model exists only in the runtime that trained it.

---

## 5. Rendering and final metrics

`render.py` renders the test split next to the ground truth; `metrics.py` computes PSNR, SSIM and LPIPS on the saved frames. The last cell reports the storage per rendered frame.

In [ ]:
# Render the test split and compute the final metrics with the repository scripts
import os

os.chdir(REPO_DIR)

!python render.py -m "{MODEL_OUTPUT_PATH}" --mode render

!python metrics.py -m "{MODEL_OUTPUT_PATH}"

In [ ]:
from pathlib import Path

OUTPUT_DIR = MODEL_OUTPUT_PATH
output_path = Path(OUTPUT_DIR)

if not output_path.exists():
    raise FileNotFoundError(f"❌ Folder {OUTPUT_DIR} does not exist. Make sure the path is correct.")

rendered_images = list(output_path.glob("**/renders/*.png")) + list(output_path.glob("**/renders/*.jpg"))
num_frames = len(rendered_images)

if num_frames == 0:
    print("⚠️ Warning: No images found in the 'renders' folder. Setting num_frames = 1 to avoid errors.")
    num_frames = 1

total_size_bytes = sum(f.stat().st_size for f in output_path.glob('**/*') if f.is_file())
total_size_mb = total_size_bytes / (1024 * 1024)

storage_per_frame_mb = (total_size_bytes / num_frames) / (1024 * 1024)

print("=" * 45)
print(" 📦 MODEL AND FRAME STORAGE CALCULATION ")
print("=" * 45)
print(f"• Total frames detected    : {num_frames}")
print(f"• Total model storage      : {total_size_mb:.2f} MB")
print(f"• **Storage per frame**    : {storage_per_frame_mb:.4f} MB / frame")
print("=" * 45)

## 6. Real-time streaming viewer

Deformable-3DGS has no built-in viewer suited to Colab (its real-time viewer is a separate project that needs OpenGL/X11). Rendering therefore happens server-side with the training rasterizer; each frame is JPEG-encoded and streamed to the browser over WebSocket. The browser controls an orbit camera (drag to rotate, scroll to zoom) and a time slider with Play/Stop that evaluates the deformation network on the fly.

In [ ]:
!pip install -q websockets

In [ ]:
# Load the trained Deformable-3DGS model for the viewer

import os
import sys
import subprocess
import importlib
import torch

REPO_DIR = "/content/Deformable-3D-Gaussians"

os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repository: {REPO_DIR}")


SIMPLE_KNN_DIR = os.path.join(
    REPO_DIR,
    "submodules",
    "simple-knn"
)

SIMPLE_KNN_CU = os.path.join(
    SIMPLE_KNN_DIR,
    "simple_knn.cu"
)

if not os.path.isdir(SIMPLE_KNN_DIR):
    raise FileNotFoundError(
        f"simple-knn directory not found:\n{SIMPLE_KNN_DIR}"
    )


if os.path.exists(SIMPLE_KNN_CU):

    with open(SIMPLE_KNN_CU, "r") as f:
        content = f.read()

    if "#include <float.h>" not in content:

        with open(SIMPLE_KNN_CU, "w") as f:
            f.write("#include <float.h>\n" + content)

        print("✓ Added <float.h> to simple_knn.cu")

    else:
        print("✓ simple_knn.cu already patched")


simple_knn_ok = False

try:

    importlib.invalidate_caches()

    import simple_knn
    from simple_knn._C import distCUDA2

    simple_knn_ok = True

    print("✓ simple_knn is already available")

except Exception as e:

    print("⚠ simple_knn is not correctly available.")
    print(f"  Reason: {type(e).__name__}: {e}")


if not simple_knn_ok:

    print("\nRebuilding simple-knn for the current Colab environment...")

    env = os.environ.copy()

    env["TORCH_CUDA_ARCH_LIST"] = "7.5"
    env["MAX_JOBS"] = "4"

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "plyfile"
        ],
        check=True
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-build-isolation",
            "--no-cache-dir",
            "--force-reinstall",
            SIMPLE_KNN_DIR
        ],
        check=True,
        env=env
    )

    importlib.invalidate_caches()

    try:

        import simple_knn
        from simple_knn._C import distCUDA2

        print("✓ simple_knn successfully rebuilt and imported")

    except Exception as e:

        raise RuntimeError(
            "simple_knn was rebuilt but cannot be imported.\n"
            f"Error: {type(e).__name__}: {e}"
        )


from argparse import ArgumentParser, Namespace

from scene import Scene, GaussianModel, DeformModel
from gaussian_renderer import render

from arguments import (
    ModelParams,
    PipelineParams,
    get_combined_args
)

from utils.general_utils import safe_state

print("✓ Deformable 3D Gaussians modules imported successfully")


MODEL_PATH = MODEL_OUTPUT_PATH

cfg_args_path = os.path.join(
    MODEL_PATH,
    "cfg_args"
)

if not os.path.exists(cfg_args_path):

    raise FileNotFoundError(
        f"cfg_args not found in {MODEL_PATH}. "
        "Has training completed?"
    )

print(f"✓ Model found: {MODEL_PATH}")


_parser = ArgumentParser()

_model_params = ModelParams(
    _parser,
    sentinel=True
)

_pipeline_params = PipelineParams(
    _parser
)

_parser.add_argument(
    "--iteration",
    default=-1,
    type=int
)


# get_combined_args() reads the model path from sys.argv, so it is set temporarily.

_original_argv = sys.argv.copy()

try:

    sys.argv = [
        "colab",
        "-m",
        MODEL_PATH
    ]

    _known_args = get_combined_args(_parser)

finally:

    sys.argv = _original_argv


print("✓ Model configuration loaded")


dataset = _model_params.extract(
    _known_args
)

pipeline = _pipeline_params.extract(
    _known_args
)


safe_state(False)

gaussians = GaussianModel(
    dataset.sh_degree
)

scene = Scene(
    dataset,
    gaussians,
    load_iteration=_known_args.iteration,
    shuffle=False
)


deform = DeformModel(
    dataset.is_blender,
    dataset.is_6dof
)

deform.load_weights(
    dataset.model_path
)


train_cameras = scene.getTrainCameras()

if len(train_cameras) == 0:

    raise RuntimeError(
        "No training cameras were found in the loaded scene."
    )

ref_camera = train_cameras[0]


print("\n" + "=" * 60)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 60)

print(
    f"Reference camera: "
    f"{ref_camera.image_width}x{ref_camera.image_height}"
)

print(
    f"FoVx: {ref_camera.FoVx:.3f}"
)

print(
    f"FoVy: {ref_camera.FoVy:.3f}"
)

print(
    f"Gaussians in model: "
    f"{gaussians.get_xyz.shape[0]}"
)

print(
    f"Blender model: "
    f"{dataset.is_blender}"
)

print(
    f"6-DoF deformation: "
    f"{dataset.is_6dof}"
)

print("=" * 60)

In [ ]:
import math
import numpy as np
import torch
from scene.cameras import MiniCam
from utils.graphics_utils import getWorld2View2, getProjectionMatrix

class OrbitCamera:
    def __init__(self, ref_cam, radius=4.0):
        self.image_width = ref_cam.image_width
        self.image_height = ref_cam.image_height
        self.FoVx = ref_cam.FoVx
        self.FoVy = ref_cam.FoVy
        self.znear = ref_cam.znear
        self.zfar = ref_cam.zfar
        self.center = np.array([0.0, 0.0, 0.0])
        self.radius = radius
        self.azimuth = 0.0
        self.elevation = 0.2

    def orbit(self, d_az, d_el):
        self.azimuth += d_az
        self.elevation = float(np.clip(self.elevation + d_el, -1.5, 1.5))

    def zoom(self, factor):
        self.radius = float(np.clip(self.radius * factor, 0.5, 20.0))

    def _position(self):
        x = self.radius * math.cos(self.elevation) * math.sin(self.azimuth)
        y = self.radius * math.sin(self.elevation)
        z = self.radius * math.cos(self.elevation) * math.cos(self.azimuth)
        return self.center + np.array([x, y, z])

    def build_camera(self):
        eye = self._position()
        forward = (self.center - eye)
        forward /= (np.linalg.norm(forward) + 1e-8)  # camera looks toward the center; Blender local +Z = -forward
        world_up = np.array([0.0, 1.0, 0.0])
        right = np.cross(forward, world_up)
        right /= (np.linalg.norm(right) + 1e-8)
        cam_up = np.cross(right, forward)

        # Camera-to-world in Blender convention: columns = [right, up, -forward]
        c2w = np.eye(4, dtype=np.float64)
        c2w[:3, 0] = right
        c2w[:3, 1] = cam_up
        c2w[:3, 2] = -forward
        c2w[:3, 3] = eye

        # Same conversion as dataset_readers.py for Blender/D-NeRF cameras
        matrix = np.linalg.inv(c2w)
        R = -np.transpose(matrix[:3, :3])
        R[:, 0] = -R[:, 0]
        T = -matrix[:3, 3]

        world_view_transform = torch.tensor(getWorld2View2(R.astype(np.float32), T.astype(np.float32))).transpose(0, 1).cuda()
        projection_matrix = getProjectionMatrix(
            znear=self.znear, zfar=self.zfar, fovX=self.FoVx, fovY=self.FoVy
        ).transpose(0, 1).cuda()
        full_proj_transform = world_view_transform.unsqueeze(0).bmm(
            projection_matrix.unsqueeze(0)
        ).squeeze(0)

        return MiniCam(
            width=self.image_width, height=self.image_height,
            fovy=self.FoVy, fovx=self.FoVx,
            znear=self.znear, zfar=self.zfar,
            world_view_transform=world_view_transform,
            full_proj_transform=full_proj_transform
        )

orbit_cam = OrbitCamera(ref_camera, radius=4.0)
display("OrbitCamera ready (Blender camera convention).")

In [ ]:
import io
from PIL import Image

background_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
background = torch.tensor(background_color, dtype=torch.float32, device="cuda")

@torch.no_grad()
def render_frame_jpeg(orbit_cam, t_normalized, quality=80):
    cam = orbit_cam.build_camera()
    xyz = gaussians.get_xyz
    fid = torch.tensor([t_normalized], dtype=torch.float32, device="cuda")
    time_input = fid.unsqueeze(0).expand(xyz.shape[0], -1)
    d_xyz, d_rotation, d_scaling = deform.step(xyz.detach(), time_input)
    render_pkg = render(cam, gaussians, pipeline, background, d_xyz, d_rotation, d_scaling, dataset.is_6dof)
    image = render_pkg["render"]
    image = (image.clamp(0, 1) * 255).byte().permute(1, 2, 0).cpu().numpy()
    buf = io.BytesIO()
    Image.fromarray(image).save(buf, format="JPEG", quality=quality)
    return buf.getvalue()

_test = render_frame_jpeg(orbit_cam, 0.0)
display(f"Test frame: {len(_test)/1024:.1f} KB")

In [ ]:
import asyncio, json, threading
import websockets

render_lock = threading.Lock()
current_time = 0.0
TARGET_FPS = 15

async def handle_client(websocket):
    print("Client connected.")
    try:
        async def receive_controls():
            global current_time
            async for message in websocket:
                try:
                    msg = json.loads(message)
                except json.JSONDecodeError:
                    continue
                t = msg.get("type")
                if t == "orbit":
                    orbit_cam.orbit(msg.get("d_azimuth", 0.0), msg.get("d_elevation", 0.0))
                elif t == "zoom":
                    orbit_cam.zoom(msg.get("factor", 1.0))
                elif t == "time":
                    current_time = float(msg.get("value", 0.0))

        async def stream_frames():
            while True:
                with render_lock:
                    frame = render_frame_jpeg(orbit_cam, current_time)
                await websocket.send(frame)
                await asyncio.sleep(1.0 / TARGET_FPS)

        await asyncio.gather(receive_controls(), stream_frames())
    except websockets.exceptions.ConnectionClosed:
        print("Client disconnected.")

async def start_server():
    async with websockets.serve(handle_client, "localhost", 8765, max_size=None):
        print("WebSocket server listening on ws://localhost:8765")
        await asyncio.Future()

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_server())

threading.Thread(target=run_server, daemon=True).start()
display("Server started in the background.")

In [ ]:
import subprocess
from IPython.display import display
from google.colab.output import eval_js

# Serve viewer_client.html (written by the next cell) on port 8000
http_proc = subprocess.Popen(
    ["python", "-m", "http.server", "8000", "--directory", "/content/Deformable-3D-Gaussians"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
display("HTTP server started on port 8000.")

# Colab proxies over HTTPS, so the browser must use wss:// through the proxied hostname.
ws_proxy_url = eval_js("google.colab.kernel.proxyPort(8765)")
ws_url = ws_proxy_url.rstrip("/").replace("https://", "wss://")
display(f"WebSocket URL to embed in the client: {ws_url}")

In [ ]:
from IPython.display import display

viewer_html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Deformable 3D Gaussians Live Viewer</title>
<style>
body{{margin:0;background:#111;color:#eee;font-family:sans-serif;}}
#canvas{{display:block;cursor:grab;}}
#hud{{position:fixed;top:10px;left:10px;padding:8px 12px;background:rgba(0,0,0,.5);border-radius:6px;}}
#playBtn{{background:#2a7;border:none;color:white;font-size:14px;padding:4px 10px;border-radius:4px;cursor:pointer;margin-bottom:6px;}}
#playBtn.stop{{background:#c33;}}
</style></head><body>
<canvas id="canvas"></canvas>
<div id="hud">
  <div>Drag = orbit • Scroll = zoom</div>
  <button id="playBtn">▶ Play</button>
  <div>t = <span id="tValue">0.00</span></div>
  <input type="range" id="timeSlider" min="0" max="1" step="0.01" value="0">
</div>
<script>
const canvas = document.getElementById('canvas');
const ctx = canvas.getContext('2d');
const img = new Image();
let dragging=false, lastX=0, lastY=0;
const ws = new WebSocket("{ws_url}");
ws.binaryType = "blob";
ws.onopen = () => console.log("WebSocket connected");
ws.onerror = (e) => console.error("WebSocket error", e);
ws.onmessage = (e) => {{
  const url = URL.createObjectURL(e.data);
  img.onload = () => {{ canvas.width=img.width; canvas.height=img.height; ctx.drawImage(img,0,0); URL.revokeObjectURL(url); }};
  img.src = url;
}};
canvas.addEventListener('mousedown', e => {{ dragging=true; lastX=e.clientX; lastY=e.clientY; canvas.style.cursor='grabbing'; }});
window.addEventListener('mouseup', () => {{ dragging=false; canvas.style.cursor='grab'; }});
window.addEventListener('mousemove', e => {{
  if(!dragging) return;
  const dx=(e.clientX-lastX)*0.005, dy=(e.clientY-lastY)*0.005;
  lastX=e.clientX; lastY=e.clientY;
  ws.send(JSON.stringify({{type:"orbit", d_azimuth:-dx, d_elevation:dy}}));
}});
canvas.addEventListener('wheel', e => {{
  e.preventDefault();
  ws.send(JSON.stringify({{type:"zoom", factor: e.deltaY>0?1.1:0.9}}));
}}, {{passive:false}});

const slider = document.getElementById('timeSlider');
const tValue = document.getElementById('tValue');
const playBtn = document.getElementById('playBtn');

function sendTime(t) {{
  tValue.textContent = t.toFixed(2);
  ws.send(JSON.stringify({{type:"time", value: t}}));
}}

slider.addEventListener('input', () => {{
  sendTime(parseFloat(slider.value));
}});

let playing = false;
let playInterval = null;
const PLAY_STEP = 0.01;
const PLAY_INTERVAL_MS = 50;

playBtn.addEventListener('click', () => {{
  playing = !playing;
  if (playing) {{
    playBtn.textContent = '⏸ Stop';
    playBtn.classList.add('stop');
    playInterval = setInterval(() => {{
      let t = parseFloat(slider.value) + PLAY_STEP;
      if (t > 1.0) t = 0.0;
      slider.value = t;
      sendTime(t);
    }}, PLAY_INTERVAL_MS);
  }} else {{
    playBtn.textContent = '▶ Play';
    playBtn.classList.remove('stop');
    clearInterval(playInterval);
  }}
}});
</script></body></html>"""

with open("/content/Deformable-3D-Gaussians/viewer_client.html", "w") as f:
    f.write(viewer_html)
display("Client HTML written with the proxied WebSocket URL and Play/Stop control.")

In [ ]:
from IPython.display import display
from google.colab.output import eval_js

# The HTTP server was started above; this only prints the link.
page_url = eval_js("google.colab.kernel.proxyPort(8000)")
display(f"{page_url}/viewer_client.html")

## 7. MP4 video export

Renders frames from the trained model while time goes from `t = 0` to `t = 1`, with a fixed camera or a smooth orbit, and encodes them into an MP4. The last cells render a 360° orbit around the world Z axis and display the video in Colab.

In [ ]:
# MP4 export configuration

import os
import math
import shutil
import time
import numpy as np
import torch
from PIL import Image
from pathlib import Path


VIDEO_DURATION = 5.0       # seconds
VIDEO_FPS = 30              # frames per second
NUM_FRAMES = int(VIDEO_DURATION * VIDEO_FPS)

VIDEO_OUTPUT_DIR = os.path.join(
    MODEL_OUTPUT_PATH,
    "video"
)

os.makedirs(VIDEO_OUTPUT_DIR, exist_ok=True)

FRAMES_DIR = "/content/deformable_video_frames"

if os.path.exists(FRAMES_DIR):
    shutil.rmtree(FRAMES_DIR)

os.makedirs(FRAMES_DIR, exist_ok=True)


# False = fixed camera, True = smooth 360° orbit

ENABLE_CAMERA_ORBIT = False

CAMERA_ORBITS = 1.0


VIDEO_WIDTH = ref_camera.image_width
VIDEO_HEIGHT = ref_camera.image_height


print("=" * 60)
print("VIDEO EXPORT CONFIGURATION")
print("=" * 60)

print(f"Duration       : {VIDEO_DURATION:.2f} s")
print(f"FPS            : {VIDEO_FPS}")
print(f"Frames         : {NUM_FRAMES}")
print(f"Resolution     : {VIDEO_WIDTH} x {VIDEO_HEIGHT}")
print(f"Camera orbit   : {ENABLE_CAMERA_ORBIT}")
print(f"Camera rotations: {CAMERA_ORBITS}")
print(f"Output folder  : {VIDEO_OUTPUT_DIR}")

print("=" * 60)

In [ ]:
# 360° orbit around the world Z axis while time advances from 0 to 1

import os
import math
import time
import numpy as np
import torch

from PIL import Image
from IPython.display import display

from scene.cameras import MiniCam
from utils.graphics_utils import (
    getWorld2View2,
    getProjectionMatrix
)


CENTER = np.array(
    [0.0, 0.0, 0.0],
    dtype=np.float64
)

ORBIT_RADIUS = 4.0

CAMERA_OFFSET_Z = 0.65

TOTAL_ROTATION = 2.0 * math.pi

START_ANGLE = 0.0

IMAGE_ROTATION = 0.0


def create_orbit_camera_z(angle):

    eye = CENTER + np.array(
        [
            ORBIT_RADIUS * math.cos(angle),
            ORBIT_RADIUS * math.sin(angle),
            CAMERA_OFFSET_Z
        ],
        dtype=np.float64
    )


    forward = (
        CENTER - eye
    )

    forward /= (
        np.linalg.norm(forward)
        + 1e-12
    )


    # World Z is the camera-up reference

    world_up = np.array(
        [0.0, 0.0, 1.0],
        dtype=np.float64
    )


    right = np.cross(
        forward,
        world_up
    )

    right_norm = np.linalg.norm(
        right
    )

    if right_norm < 1e-8:

        raise RuntimeError(
            "Degenerate camera orientation."
        )

    right /= right_norm


    cam_up = np.cross(
        right,
        forward
    )

    cam_up /= (
        np.linalg.norm(cam_up)
        + 1e-12
    )


    # Blender camera convention, then the repository conversion

    c2w = np.eye(
        4,
        dtype=np.float64
    )

    c2w[:3, 0] = right
    c2w[:3, 1] = cam_up
    c2w[:3, 2] = -forward
    c2w[:3, 3] = eye


    matrix = np.linalg.inv(c2w)

    R = -np.transpose(
        matrix[:3, :3]
    )

    R[:, 0] = -R[:, 0]

    T = -matrix[:3, 3]


    world_view_transform = torch.tensor(
        getWorld2View2(
            R.astype(np.float32),
            T.astype(np.float32)
        )
    ).transpose(
        0,
        1
    ).cuda()


    projection_matrix = (
        getProjectionMatrix(
            znear=ref_camera.znear,
            zfar=ref_camera.zfar,
            fovX=ref_camera.FoVx,
            fovY=ref_camera.FoVy
        )
        .transpose(
            0,
            1
        )
        .cuda()
    )


    full_proj_transform = (
        world_view_transform
        .unsqueeze(0)
        .bmm(
            projection_matrix
            .unsqueeze(0)
        )
        .squeeze(0)
    )


    return MiniCam(
        width=ref_camera.image_width,
        height=ref_camera.image_height,
        fovy=ref_camera.FoVy,
        fovx=ref_camera.FoVx,
        znear=ref_camera.znear,
        zfar=ref_camera.zfar,
        world_view_transform=world_view_transform,
        full_proj_transform=full_proj_transform
    )


@torch.no_grad()
def render_video_frame(
    frame_index,
    total_frames
):

    if total_frames <= 1:

        t = 0.0

    else:

        t = (
            frame_index /
            (total_frames - 1)
        )


    angle = (
        START_ANGLE
        +
        TOTAL_ROTATION * t
    )


    cam = create_orbit_camera_z(
        angle
    )


    # Query the deformation field at time t

    xyz = gaussians.get_xyz

    fid = torch.tensor(
        [t],
        dtype=torch.float32,
        device="cuda"
    )

    time_input = (
        fid
        .unsqueeze(0)
        .expand(
            xyz.shape[0],
            -1
        )
    )


    d_xyz, d_rotation, d_scaling = (
        deform.step(
            xyz.detach(),
            time_input
        )
    )


    render_pkg = render(
        cam,
        gaussians,
        pipeline,
        background,
        d_xyz,
        d_rotation,
        d_scaling,
        dataset.is_6dof
    )


    image = render_pkg["render"]


    image = (
        image
        .clamp(0, 1)
        .mul(255)
        .byte()
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )


    return image


display("=" * 60)
display("360° ORBIT — ATTEMPT 3")
display("=" * 60)

display("Rotation axis: WORLD Z")
display("Orbit plane: XY")
display(
    f"Radius: {ORBIT_RADIUS}"
)
display(
    f"Z offset: {CAMERA_OFFSET_Z}"
)

display("=" * 60)


start_time = time.time()


for frame_idx in range(
    NUM_FRAMES
):

    frame = render_video_frame(
        frame_index=frame_idx,
        total_frames=NUM_FRAMES
    )


    image = Image.fromarray(
        frame
    )

    image = image.rotate(
        IMAGE_ROTATION,
        expand=True
    )


    frame_path = os.path.join(
        FRAMES_DIR,
        f"frame_{frame_idx:05d}.png"
    )

    image.save(
        frame_path
    )


    elapsed = (
        time.time() -
        start_time
    )

    progress = (
        (frame_idx + 1)
        /
        NUM_FRAMES
        *
        100
    )

    angle_deg = (
        360.0
        *
        frame_idx
        /
        max(
            NUM_FRAMES - 1,
            1
        )
    )

    t = (
        frame_idx
        /
        max(
            NUM_FRAMES - 1,
            1
        )
    )

    avg_time = (
        elapsed /
        (frame_idx + 1)
    )

    remaining = (
        avg_time *
        (
            NUM_FRAMES
            -
            frame_idx
            -
            1
        )
    )


    display(
        f"Frame {frame_idx + 1}/{NUM_FRAMES} | "
        f"{progress:.1f}% | "
        f"Z orbit: {angle_deg:.1f}° | "
        f"t={t:.3f} | "
        f"ETA={remaining:.1f}s"
    )


total_time = (
    time.time() -
    start_time
)

display("=" * 60)
display("Z-AXIS ORBIT COMPLETED")
display("=" * 60)

display(
    f"Total time: {total_time:.2f}s"
)

display(
    f"Average/frame: "
    f"{total_time / NUM_FRAMES:.3f}s"
)

display(
    f"Frames saved: {FRAMES_DIR}"
)

display("=" * 60)

In [ ]:
# Encode the PNG frames into an H.264 MP4

VIDEO_PATH = os.path.join(
    VIDEO_OUTPUT_DIR,
    "deformable_3d_gaussians_5s.mp4"
)

print("=" * 60)
print("ENCODING MP4")
print("=" * 60)

ffmpeg_command = [
    "ffmpeg",
    "-y",

    "-framerate",
    str(VIDEO_FPS),

    "-i",
    os.path.join(
        FRAMES_DIR,
        "frame_%05d.png"
    ),

    "-c:v",
    "libx264",

    "-pix_fmt",
    "yuv420p",

    "-crf",
    "18",

    "-preset",
    "medium",

    VIDEO_PATH
]


result = subprocess.run(
    ffmpeg_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)


if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "FFmpeg failed while creating the MP4."
    )


print("✓ MP4 successfully created")
print()
print(f"Video: {VIDEO_PATH}")

video_size_mb = (
    os.path.getsize(VIDEO_PATH)
    / (1024 * 1024)
)

print(
    f"Size : {video_size_mb:.2f} MB"
)

print("=" * 60)

In [ ]:
# Display the MP4 inside Colab

from IPython.display import Video, display

display(
    Video(
        VIDEO_PATH,
        embed=True,
        width=VIDEO_WIDTH,
        height=VIDEO_HEIGHT
    )
)